# Hakam — full-corpus feature extraction (Colab GPU)

Produces the **frozen baseline** embeddings for all 8,297 labelled clips.

The diagnostic on 2 Sep 2026 found that frozen Kinetics features encode camera framing
(probe 0.94) but barely encode the foul itself (card probe ~0.47–0.56). So these
embeddings are **not** the final model — they are the baseline that Experiment 1
compares fine-tuning against, which §7 of the brief requires anyway.

### NDA rule, enforced by the structure of this notebook

The dataset is downloaded **into the ephemeral runtime** and dies with it. Only the
embedding cache (~25 MB, no video content) is copied to Drive. Never add a cell that
writes `data/` to Drive.

**Runtime → Change runtime type → GPU** before running anything.

In [ ]:
# 1. Confirm a GPU is actually attached. On CPU this notebook takes ~3 hours
#    instead of ~30 minutes, so fail loudly rather than discovering it later.
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun."
)
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Get the code. If the repo is private, use a personal access token:
#      !git clone https://<TOKEN>@github.com/FerasMad/hakam.git
#    Do not paste a token into a cell you intend to commit.
%cd /content
!git clone https://github.com/FerasMad/hakam.git 2>/dev/null || (cd hakam && git pull)
%cd /content/hakam

In [ ]:
# 3. Dependencies. Colab already ships torch + torchvision built against its CUDA,
#    so install only what is missing rather than letting pip resolve torch again.
!pip install -q transformers SoccerNet opencv-python-headless

import transformers, cv2
print("transformers", transformers.__version__, "| cv2", cv2.__version__)

In [ ]:
# 4. Download the dataset into the runtime. ~3.3 GB, a few minutes on Colab's link.
#
#    The password comes from Colab Secrets, never from a cell. Set it up once:
#      key icon in the left sidebar -> Add new secret
#      name:  SOCCERNET_PASSWORD
#      value: <the NDA password>
#      then toggle notebook access on
#
#    It lives in your Google account: never in the notebook, never in git, never in
#    a cell output. getpass is only the fallback if the secret is missing.
from SoccerNet.Downloader import SoccerNetDownloader

try:
    from google.colab import userdata

    password = userdata.get("SOCCERNET_PASSWORD")
except Exception:
    from getpass import getpass

    password = getpass("SOCCERNET_PASSWORD secret not found - enter password: ")

downloader = SoccerNetDownloader(LocalDirectory="/content/hakam/data/mvfouls")
downloader.password = password
downloader.downloadDataTask(task="mvfouls", split=["train", "valid", "test"])

In [ ]:
# 5. Normalise the layout.
#
#    The zips extract FLAT: every split writes action_0, action_1, ... into the same
#    directory, so splits silently overwrite each other. Each split must sit in its
#    own folder. This is the single most destructive gotcha in the dataset.
from pathlib import Path
import zipfile

root = Path("/content/hakam/data/mvfouls")
for split in ["Train", "Valid", "Test"]:
    target = root / split
    target.mkdir(parents=True, exist_ok=True)
    archive = next(root.glob(f"{split.lower()}*.zip"), None) or next(
        root.glob(f"{split}*.zip"), None
    )
    if archive is not None:
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(target)
        print(f"{split}: extracted {archive.name}")

for split in ["Train", "Valid", "Test"]:
    d = root / split
    print(f"{split:6} annotations.json={(d / 'annotations.json').exists()}  "
          f"actions={len(list(d.glob('action_*')))}")

In [ ]:
# 6. Verify the splits match the official counts before spending GPU time.
import sys
sys.path.insert(0, "/content/hakam")

from src import config
from src.data.annotations import load_split, verify_clips_exist

for split in ["train", "valid", "test"]:
    actions, clips = load_split(config.DATA_ROOT / "mvfouls" / config.SPLIT_DIRS[split])
    missing = verify_clips_exist(clips)
    print(f"{split:6} actions={len(actions):>5} (official {config.SPLIT_SIZES[split]})  "
          f"clips={len(clips):>5}  missing={len(missing)}")

In [ ]:
# 7. Extract. Both backbones in one go - the second costs only minutes while the
#    runtime is warm, and it pre-pays Experiment 1's backbone comparison.
#
#    Window is 43-107 (2.56 s), reproducing VideoMAE's 16-frames-at-stride-4
#    pretraining. The published 63-87 window spans 0.96 s and yields near-duplicate
#    frames; it lost on every measurement.
!python scripts/extract_all.py \
    --backbones videomae_small videomae_base \
    --splits train valid test \
    --batch-size 32

In [ ]:
# 8. Copy ONLY the embedding cache to Drive. No video, ever.
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive")
dest = Path("/content/drive/MyDrive/hakam/features_cache")
dest.mkdir(parents=True, exist_ok=True)

for f in sorted(Path("/content/hakam/features_cache").glob("*")):
    shutil.copy2(f, dest / f.name)
    print(f"{f.name}  {f.stat().st_size / 1e6:.1f} MB")

print(f"\ncopied to {dest}")

## Next

Download `features_cache/` from Drive into the local repo, then train the cascade heads
(Cycle 1). Experiments come in Cycle 2 — fine-tuning is the one that matters, since the
diagnostic showed frozen features carry only weak signal about the foul itself.

Fine-tuning trains the backbone, so it needs decoded **frames**, not this cache. It runs
in its own notebook.